要运行此程序，请在 **AMD Dev Cloud** 上按“*运行*”并按“*运行全部*”！
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> 如果您需要帮助，请加入 Discord + ⭐ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</a> </i> ⭐
</div>

要在本地设备上安装 Unsloth，请按照 [our guide](https://unsloth.ai/docs/get-started/install) 操作。此笔记本已获得许可 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)。

您将学习如何执行 [data prep](#Data)、如何执行 [train](#Train)、如何执行 [run the model](#Inference) 以及如何保存它

### 消息

隆重推出 **Unsloth Studio** - 一个新的开源、无代码 Web UI，用于训练和运行法学硕士。 [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<表><tr>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~% 2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fupload s%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia% 26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&宽度=376&dpr=3&质量=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>训练模型</b> - 无需代码</sub></td>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Z q%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26toke n%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&宽度=376&dpr=3&质量=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio 聊天 UI"></a><br><sub><b>在 Mac、Windows 和 Linux 上运行 GGUF 模型</b></sub></td>
</tr></表>

训练 MoE - DeepSeek、GLM、Qwen 和 gpt-oss 速度提高 12 倍，VRAM 减少 35%。 [Blog](https://unsloth.ai/docs/new/faster-moe)

超长上下文强化学习现已推出，上下文窗口增加了 7 倍！ [Blog](https://unsloth.ai/docs/new/grpo-long-context)

强化学习新功能：[FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

请访问我们的文档以获取所有 [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) 和 [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks)。

### 安装

In [ ]:
%%bash
python -m pip install -qU uv --root-user-action=ignore

ROCM_TAG="$({ command -v amd-smi >/dev/null 2>&1 && amd-smi version 2>/dev/null | awk -F'ROCm version: ' 'NF>1{split($2,a,"."); print "rocm"a[1]"."a[2]; ok=1; exit} END{exit !ok}'; } || { [ -r /opt/rocm/.info/version ] && awk -F. '{print "rocm"$1"."$2; exit}' /opt/rocm/.info/version; } || { command -v hipconfig >/dev/null 2>&1 && hipconfig --version 2>/dev/null | awk -F': *' '/HIP version/{split($2,a,"."); print "rocm"a[1]"."a[2]; ok=1; exit} END{exit !ok}'; } || { command -v dpkg-query >/dev/null 2>&1 && ver="$(dpkg-query -W -f='${Version}\n' rocm-core 2>/dev/null)" && [ -n "$ver" ] && awk -F'[.-]' '{print "rocm"$1"."$2; exit}' <<<"$ver"; } || { command -v rpm >/dev/null 2>&1 && ver="$(rpm -q --qf '%{VERSION}\n' rocm-core 2>/dev/null)" && [ -n "$ver" ] && awk -F'[.-]' '{print "rocm"$1"."$2; exit}' <<<"$ver"; })"
[ -n "$ROCM_TAG" ] || { echo "Could not detect ROCm. Install ROCm first or set ROCM_TAG manually."; exit 1; }
case "$ROCM_TAG" in
  rocm6.[0-4]|rocm7.[02]) T="$ROCM_TAG" ;;
  rocm6.*) T="rocm6.4" ;;
  *) T="rocm7.1" ;;
esac
pip install bitsandbytes
PYTORCH_INDEX_URL="https://download.pytorch.org/whl/${T}"
uv pip install --system -U --force-reinstall \
    torch torchvision torchaudio triton-rocm \
    --index-url "$PYTORCH_INDEX_URL"
uv pip install --system cut-cross-entropy torchao --no-deps
uv pip install --system -U --no-deps "unsloth[amd]" "unsloth_zoo[amd]"
uv pip install --system --no-deps -r "$(python -c 'import pathlib,site;print(next(p for r in [*site.getsitepackages(),site.getusersitepackages()] if (p:=pathlib.Path(r,"studio/backend/requirements/no-torch-runtime.txt")).exists()))')" torchao
uv pip install --system --no-deps -U "tokenizers>=0.22.0,<=0.23.0"


In [ ]:
!uv pip install --system -qqq sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer "transformers==4.56.2"
!uv pip install --system -qqq --no-deps accelerate peft "trl==0.22.2"


### 不懒惰

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # 选择任何一个！我们内部自动支持 RoPE Scaling！
dtype = None # 没有用于自动检测。 Float16 适用于 Tesla T4、V100，Bfloat16 适用于 Ampere+
load_in_4bit = True # 使用 4 位量化来减少内存使用。可能是假的。

# 我们支持 4 位预量化模型，下载速度提高 4 倍 + 无 OOM。
fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/llama-2-13b-bnb-4bit",
    "unsloth/codellama-34b-bnb-4bit",
    "unsloth/tinyllama-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit", # 新的谷歌 6 万亿代币模型速度提高了 2.5 倍！
    "unsloth/gemma-2b-bnb-4bit",
] # 更多模型请访问 https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit", # 选择任意！例如 teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # 门控模型的 HF 令牌
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.

config.json:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

==((====))==  Unsloth: Fast Mistral patching release 2024.5
   \\   /|    GPU: Tesla T4. Max memory: 14.748 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.3.0+cu121. CUDA = 7.5. CUDA Toolkit = 12.1.
\        /    Bfloat16 = FALSE. Xformers = 0.0.26.post1. FA = False.
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth

model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/137k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565

我们现在添加 LoRA 适配器，因此我们只需要更新所有参数的 1 到 10%！

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # 选择任意大于 0 的数字！建议 8、16、32、64、128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # 支持任意，但 = 0 已优化
    bias = "none",    # 支持任何，但=“无”已优化
    # [新]“unsloth”使用的 VRAM 减少了 30%，适合 2 倍大的批量大小！
    use_gradient_checkpointing = "unsloth", # 对于很长的上下文来说是真实的或“不懒惰的”
    random_state = 3407,
    use_rslora = False,  # 我们支持排名稳定的 LoRA
    loftq_config = None, # 和阁楼Q
)

Unsloth 2024.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


<a名称=“数据”></a>
### 数据准备
我们现在使用“ChatML”格式进行对话风格微调。我们在 ShareGPT 风格中使用 [Open Assistant conversations](https://huggingface.co/datasets/philschmid/guanaco-sharegpt-style)。 ChatML 呈现多轮对话，如下所示：

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What's the capital of France?<|im_end|>
<|im_start|>assistant
Paris.
```

**[注意]** 要仅在完成情况下进行训练（忽略用户的输入），请阅读我们的文档 [here](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide#training-on-completions-only-masking-out-inputs)

我们使用“get_chat_template”函数来获取正确的聊天模板。我们支持“zephyr、chatml、mistral、llama、alpaca、vicuna、vicuna_old”和我们自己优化的“unsloth”模板。

通常我们必须训练`<|im_start|>`和`<|im_end|>`。我们将 `<|im_end|>` 映射为 EOS 代币，并保留 `<|im_start|>` 不变。这不需要额外的令牌的额外训练。

注意 ShareGPT 使用 `{"from": " human", "value" : "Hi"}` 而不是 `{"role": "user", "content" : "Hi"}`，所以我们使用 `mapping` 来映射它。

对于小说写作等文本补全，请尝试此 [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb)。

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # 支持 zephyr、chatml、mistral、llama、alpaca、vicuna、vicuna_old、unsloth
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # 分享GPT风格
    map_eos_token = True, # 将 <|im_end|> 映射到 </s>
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

from datasets import load_dataset
dataset = load_dataset("philschmid/guanaco-sharegpt-style", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

Unsloth: Will map <|im_end|> to EOS = </s>.
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565

Generating train split:   0%|          | 0/9033 [00:00<?, ? examples/s]

Map:   0%|          | 0/9033 [00:00<?, ? examples/s]

让我们通过打印第 5 个元素来看看“ChatML”格式是如何工作的

In [ ]:
dataset[5]["conversations"]

[{'from': 'human',
  'value': 'What is the typical wattage of bulb in a lightbox?'},
 {'from': 'gpt',
  'value': 'The typical wattage of a bulb in a lightbox is 60 watts, although domestic LED bulbs are normally much lower than 60 watts, as they produce the same or greater lumens for less wattage than alternatives. A 60-watt Equivalent LED bulb can be calculated using the 7:1 ratio, which divides 60 watts by 7 to get roughly 9 watts.'},
 {'from': 'human',
  'value': 'Rewrite your description of the typical wattage of a bulb in a lightbox to only include the key points in a list format.'}]

In [ ]:
print(dataset[5]["text"])

<|im_start|>user
What is the typical wattage of bulb in a lightbox?<|im_end|>
<|im_start|>assistant
The typical wattage of a bulb in a lightbox is 60 watts, although domestic LED bulbs are normally much lower than 60 watts, as they produce the same or greater lumens for less wattage than alternatives. A 60-watt Equivalent LED bulb can be calculated using the 7:1 ratio, which divides 60 watts by 7 to get roughly 9 watts.<|im_end|>
<|im_start|>user
Rewrite your description of the typical wattage of a bulb in a lightbox to only include the key points in a list format.<|im_end|>



如果您想制作自己的聊天模板，这也是可能的！您必须使用 Jinja 模板机制。我们提供了自己的“Unsloth 模板”的精简版本，我们发现它更加高效，并利用了 ChatML、Zephyr 和 Alpaca 风格。

有关 [our wiki page!](https://github.com/unslothai/unsloth/wiki#chat-templates) 上聊天模板的更多信息

In [ ]:
unsloth_template = \
    "{{ bos_token }}"\
    "{{ 'You are a helpful assistant to the user\n' }}"\
    "{% for message in messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ '>>> User: ' + message['content'] + '\n' }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ '>>> Assistant: ' + message['content'] + eos_token + '\n' }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}"\
        "{{ '>>> Assistant: ' }}"\
    "{% endif %}"
unsloth_eos_token = "eos_token"

if False:
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = (unsloth_template, unsloth_eos_token,), # 您必须提供模板和 EOS 代币
        mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # 分享GPT风格
        map_eos_token = True, # 将 <|im_end|> 映射到 </s>
    )

<a name="火车"></a>
### 训练模型
现在让我们训练我们的模型。我们执行 60 个步骤来加快速度，但您可以设置“num_train_epochs=1”以进行完整运行，并关闭“max_steps=None”。我们还支持`DPOTrainer`和`GRPOTrainer`用于强化学习！

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # 可以使短序列的训练速度提高 5 倍。
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # 使用 TrackIO/WandB 等
    ),
)

/usr/local/lib/python3.10/dist-packages/multiprocess/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Map (num_proc=2):   0%|          | 0/9033 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


In [ ]:
# @title 显示当前内存统计信息
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.748 GB.
4.52 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 9,033 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 60
 "-____-"     Number of trainable parameters = 41,943,040


Step,Training Loss
1,1.645400
2,1.203700
3,1.153700
4,1.208300
5,1.135200
6,1.074600
7,0.829600
8,1.294600
9,1.255400
10,1.066200


In [ ]:
# @title 显示最终内存和时间统计数据
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

870.2993 seconds used for training.
14.5 minutes used for training.
Peak reserved memory = 6.289 GB.
Peak reserved memory for training = 1.769 GB.
Peak reserved memory % of max memory = 42.643 %.
Peak reserved memory for training % of max memory = 11.995 %.


<a name="推理"></a>
### 推论
让我们运行模型吧！由于我们使用的是“ChatML”，因此请使用“apply_chat_template”并将“add_ Generation_prompt”设置为“True”进行推理。

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # 支持 zephyr、chatml、mistral、llama、alpaca、vicuna、vicuna_old、unsloth
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # 分享GPT风格
    map_eos_token = True, # 将 <|im_end|> 映射到 </s>
)

FastLanguageModel.for_inference(model) # 使本机推理速度提高 2 倍

messages = [
    {"from": "human", "value": "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # 必须添加生成
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


['<|im_start|>user\nContinue the fibonacci sequence: 1, 1, 2, 3, 5, 8,<|im_end|> \n<|im_start|>assistant\nThe next number in the Fibonacci sequence is 13.\n\nThe Fibonacci sequence is a series of numbers in which each number is the sum of the two preceding ones, starting from 0 and 1. The sequence goes: 0, 1, 1, ']

您还可以使用“TextStreamer”进行连续推理 - 这样您就可以逐个令牌查看生成令牌，而不是一直等待！

In [ ]:
FastLanguageModel.for_inference(model) # 使本机推理速度提高 2 倍

messages = [
    {"from": "human", "value": "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # 必须添加生成
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128, use_cache = True)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<|im_start|>user
Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,<|im_end|> 
<|im_start|>assistant
The next number in the Fibonacci sequence is 13.

The Fibonacci sequence is a series of numbers in which each number is the sum of the two preceding ones, starting from 0 and 1. The sequence goes: 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2


<a name="保存"></a>
### 保存、加载微调模型
要将最终模型保存为 LoRA 适配器，请使用 Hugging Face 的 `push_to_hub` 进行在线保存，或使用 `save_pretrained` 进行本地保存。

**[注意]** 这仅保存 LoRA 适配器，而不是完整模型。要保存到 16 位或 GGUF，请向下滚动！

In [ ]:
model.save_pretrained("mistral_v0_lora")  # 本地储蓄
tokenizer.save_pretrained("mistral_v0_lora")
# model.push_to_hub("your_name/mistral_v0_lora", token = "YOUR_HF_TOKEN") # 在线保存
# tokenizer.push_to_hub("your_name/mistral_v0_lora", token = "YOUR_HF_TOKEN") # 在线保存

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

现在，如果您想加载我们刚刚保存用于推理的 LoRA 适配器，请将“False”设置为“True”：

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "mistral_v0_lora", # 您用于训练的模型
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # 使本机推理速度提高 2 倍

messages = [
    {"from": "human", "value": "What is a famous tall tower in Paris?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # 必须添加生成
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128, use_cache = True)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<|im_start|>user
What is a famous tall tower in Paris?<|im_end|> 
<|im_start|>assistant
The Eiffel Tower is a famous tall tower in Paris. It is one of the most recognizable landmarks in the world and is a popular tourist destination. The tower was built in 1889 for the World's Fair and is named after Gustave Eiffel, the engineer who designed and built it. The tower stands at a height of 324 meters (1,063 feet) and is made of iron. It is located on the Champ de Mars, a large public park in Paris.<|im_end|>


您还可以使用 Hugging Face 的“AutoPeftModelForCausalLM”。仅当您未安装“unsloth”时才使用此选项。它可能会慢得令人绝望，因为不支持“4bit”模型下载，而且 Unsloth 的**推理速度快了 2 倍**。

In [ ]:
if False:
    # 我强烈不建议 - 如果可能的话使用 Unsloth
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer

    model = AutoPeftModelForCausalLM.from_pretrained(
        "mistral_v0_lora",  # 您用于训练的模型
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("mistral_v0_lora")

### 保存为 VLLM 的 float16

我们还支持直接保存到`float16`。对于 float16 选择“merged_16bit”，对于 int4 选择“merged_4bit”。我们还允许“lora”适配器作为后备。使用 `push_to_hub_merged` 上传到您的 Hugging Face 帐户！您可以前往 https://huggingface.co/settings/tokens 获取您的个人代币。有关更多部署选项，请参阅 [our docs](https://unsloth.ai/docs/basics/inference-and-deployment)。

In [ ]:
# 合并到16位
if False: model.save_pretrained_merged("mistral_v0_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/mistral_v0_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# 合并到4bit
if False: model.save_pretrained_merged("mistral_v0_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/mistral_v0_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# 只需 LoRA 适配器
if False:
    model.save_pretrained("mistral_v0_lora")
    tokenizer.save_pretrained("mistral_v0_lora")
if False:
    model.push_to_hub("HF_USERNAME/mistral_v0_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/mistral_v0_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp 转换
要保存到 `GGUF` / `llama.cpp`，我们现在原生支持它！我们克隆 `llama.cpp` 并默认将其保存到 `q8_0`。我们允许像“q4_k_m”这样的所有方法。使用`save_pretrained_gguf`进行本地保存，使用`push_to_hub_gguf`上传到HF。

一些支持的量化方法（完整列表在我们的 [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf) 上）：
* `q8_0` - 快速转换。资源使用率较高，但总体上可以接受。
* `q4_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q4_K。
* `q5_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q5_K。

In [ ]:
# 保存到8位Q8_0
if False: model.save_pretrained_gguf("mistral_v0_finetune", tokenizer,)
if False: model.push_to_hub_gguf("HF_USERNAME/mistral_v0_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# 保存到16位GGUF
if False: model.save_pretrained_gguf("mistral_v0_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/mistral_v0_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# 保存到 q4_k_m GGUF
if False: model.save_pretrained_gguf("mistral_v0_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/mistral_v0_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

我们就完成了！如果您对 Unsloth 有任何疑问，我们有 [Discord](https://discord.gg/unsloth) 频道！如果您发现任何错误或想要随时更新最新的 LLM 内容，或需要帮助、加入项目等，请随时加入我们的 Discord！

其他一些资源：
1. 希望在本地使用 Unsloth？请阅读我们的 [Installation Guide](https://unsloth.ai/docs/get-started/install)，了解有关在 Windows、Docker、AMD、Intel GPU 上安装 Unsloth 的详细信息。
2. 了解如何使用我们的 [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) 进行强化学习。
3. 阅读我们的指南和笔记本以了解 [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) 和 [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) 模型支持。
4. 浏览我们的 [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) 以查找每个型号的专用指南。
5. 需要推理方面的帮助吗？请阅读我们的 [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment)，了解有关使用 vLLM、llama.cpp、Ollama 等的详细信息。

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  如果您需要帮助，请加入 Discord + ⭐️ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</i> ⭐️

  <b>此笔记本和所有 Unsloth 笔记本均已获得 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme) 许可</b>
</div>